In [1]:
import shutil
shutil.rmtree('/content/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article', ignore_errors=True)
shutil.rmtree('/content/ml-1m', ignore_errors=True)

In [2]:
!pip install gin-config polars einops tqdm -q
!git clone -b movielens_branch https://github.com/mikhaildanilov/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article.git
%cd deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article
!mkdir -p dataset/ml-1m/raw
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip -q ml-1m.zip
!mv ml-1m/*.dat dataset/ml-1m/raw/

Cloning into 'deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 318 (delta 44), reused 19 (delta 5), pack-reused 219 (from 1)
Receiving objects: 100% (318/318), 60.88 MiB | 13.53 MiB/s, done.
Resolving deltas: 100% (149/149), done.
/content/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article


In [3]:
!sed -i 's/device = "gpu"/device = "cuda"/' config/baselines/tiger_random_ml1m.gin
!sed -i 's/t5_d_model = 128/t5_d_model = 64/' config/baselines/tiger_random_ml1m.gin
!sed -i 's/t5_num_heads = 6/t5_num_heads = 2/' config/baselines/tiger_random_ml1m.gin
!sed -i 's/t5_num_layers = 4/t5_num_layers = 2/' config/baselines/tiger_random_ml1m.gin
!sed -i 's/t5_d_ff = 1024/t5_d_ff = 256/' config/baselines/tiger_random_ml1m.gin
!sed -i 's/max_len = 200/max_len = 50/' config/baselines/tiger_random_ml1m.gin
!sed -i 's/batch_size = 256/batch_size = 64/' config/baselines/tiger_random_ml1m.gin

In [4]:
!cat config/baselines/tiger_random_ml1m.gin

# TIGER (Random IDs) on MovieLens-1M -- TIGER's encoder-decoder trained on
# randomly assigned integer codes instead of RQ-VAE Semantic IDs. Ablates
# content-based quantization: no item content is used to build the IDs.

run_baseline.model = "tiger_random"
run_baseline.dataset = "ml-1m"
run_baseline.ks = [5, 10, 50, 100]
run_baseline.seed = 42
run_baseline.device = "cuda"

# Decoding for long recommendation lists.
# gen_mode="sample": stochastic autoregressive sampling -> large candidate pool
#   (supports k = 50, 100, ...). gen_mode="beam": exact, but only for small k.
# num_samples=None auto-scales the pool to comfortably cover max(ks).
# temperature > 1 flattens the distribution (more diverse candidates).
run_baseline.gen_mode = "sample"
run_baseline.num_samples = None
run_baseline.temperature = 1.0

# Semantic-ID structure.
run_baseline.n_layers = 3
run_baseline.codebook_size = 256

# Optimisation (epochs == number of optimisation steps / batches).
# MovieLens histories are long, 

In [5]:
!cd /content/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article && git pull

Already up to date.


In [ ]:
!python -m baselines.run_config config/baselines/tiger_random_ml1m.gin

[run_config] loaded config/baselines/tiger_random_ml1m.gin
[run_baseline] model=tiger_random dataset=ml-1m split=beauty users=6040 items=3416 ks=[5, 10, 50, 100]
[TIGER-random] items=3416 train_pairs=794289 params=525312
100% 19997/20000 [09:52<00:00, 34.91it/s]
Evaluate:   0% 0/781 [00:00<?, ?it/s]
Evaluate:   0% 1/781 [00:05<1:05:30,  5.04s/it]
Evaluate:   0% 2/781 [00:10<1:05:06,  5.02s/it]
100% 19997/20000 [10:10<00:00, 34.91it/s]
Evaluate:   1% 4/781 [00:20<1:05:08,  5.03s/it]
Evaluate:   1% 5/781 [00:25<1:05:40,  5.08s/it]
Evaluate:   1% 6/781 [00:30<1:05:48,  5.09s/it]
Evaluate:   1% 7/781 [00:35<1:05:54,  5.11s/it]
Evaluate:   1% 8/781 [00:40<1:06:05,  5.13s/it]
Evaluate:   1% 9/781 [00:45<1:05:58,  5.13s/it]
Evaluate:   1% 10/781 [00:50<1:06:00,  5.14s/it]
Evaluate:   1% 11/781 [00:56<1:05:47,  5.13s/it]
Evaluate:   2% 12/781 [01:01<1:05:45,  5.13s/it]
Evaluate:   2% 13/781 [01:06<1:05:31,  5.12s/it]
Evaluate:   2% 14/781 [01:11<1:05:28,  5.12s/it]
Evaluate:   2% 15/781 [01:16